# Module 5: SARIMA and Seasonal Orders

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

[Module 4](Module_04_ARIMA_End_To_End.ipynb) showed a non seasonal model
failing on a seasonal series, and no amount of extra autoregressive terms
rescuing it. The repair is a second set of orders operating at the seasonal
lag.

Beginner [Topic 8](../../Beginner/Topic_08_Seasonality.md) and Intermediate
[Module 7](../../Intermediate/Module_07_Seasonal_Adjustment.md) both removed
the season before doing anything else. This module **models** it instead, which
keeps the seasonal uncertainty inside the forecast where it belongs.

**About 30 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]          # never fit on unfinished months


CALENDAR = pd.period_range("2019-01", "2026-04", freq="M").to_timestamp()


def counts(agency_id):
    """Monthly counts on a complete calendar, so a gap stays visible as missing."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series(d["n_uof"].values, dtype=float,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


print(f"{final['agency_id'].nunique()} agencies, {final['year_month'].nunique()} months")

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import acf, pacf, adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox


def ljung(resid, lag=12):
    return float(acorr_ljungbox(resid, lags=[lag], return_df=True)["lb_pvalue"].iloc[0])

s = np.log(counts("A012"))                 # Ashfell, the most seasonal agency
train, test = s.loc[:"2024-12"], s.loc["2025-01":"2025-12"]

## 2. What the seasonal orders mean

A SARIMA has two sets of the same three numbers.

| | Non seasonal | Seasonal |
|---|---|---|
| written | (p, d, q) | (P, D, Q) at period m |
| the lag they act on | 1, 2, 3 months | m, 2m, 3m months |
| d or D | differences against last month | differences against **the same month last year** |

For monthly data m is 12. `(0,1,1)(0,1,1)12` therefore means: difference
against last month, difference against last year, one moving average term at
lag 1, and one at lag 12. It is common enough to have a name, the **airline
model**.

## 3. Identify the seasonal orders the same way

Look at the ACF and PACF **at the seasonal lags**: 12, 24, 36. The reading
rules are the ones from Module 4, applied at those lags instead of the first
few.

In [ ]:
D12 = train.diff(12).dropna()
band = 1.96 / np.sqrt(len(D12))
a_v, p_v = acf(D12, nlags=26, fft=False), pacf(D12, nlags=26)

print(f"after one seasonal difference, band plus or minus {band:.2f}\n")
print("  lag   ACF    PACF")
for k in [1, 2, 3, 11, 12, 13, 24]:
    mark = "  <-" if abs(a_v[k]) > band or abs(p_v[k]) > band else ""
    print(f"  {k:3d}  {a_v[k]:+.2f}   {p_v[k]:+.2f}{mark}")

Something is still present at lag 12 in both, so one seasonal difference has
not finished the job and a seasonal term is needed. The ACF and PACF are close
in size there, which does not distinguish a seasonal moving average from a
seasonal autoregressive term, so the next step is to fit both and compare.

## 4. Fit the candidates

In [ ]:
cands = [((0, 1, 1), (0, 1, 1, 12)), ((1, 1, 1), (0, 1, 1, 12)),
         ((0, 1, 1), (1, 1, 1, 12)), ((0, 1, 1), (2, 1, 0, 12)),
         ((0, 1, 1), (0, 1, 2, 12)), ((0, 1, 1), (1, 1, 0, 12))]

rows = []
for o, so in cands:
    r = SARIMAX(train, order=o, seasonal_order=so, enforce_stationarity=False,
                enforce_invertibility=False).fit(disp=False)
    rows.append({"order": f"{o} {so[:3]} at 12", "AIC": round(r.aic, 1),
                 "BIC": round(r.bic, 1), "Ljung Box 24 p": round(ljung(r.resid[13:], 24), 3),
                 "parameters": len(r.params)})

pd.DataFrame(rows).sort_values("AIC").set_index("order")

The airline model wins on AIC, on BIC, on the diagnostic, and on parsimony at
the same time. That does not always happen, and when it does it is worth
saying so, because it means nothing about the choice is a judgment call.

Notice the two orders that fail the diagnostic. A model can have a respectable
AIC and still leave structure in its residuals, which is why the two columns
are both there.

## 5. Automatic selection, and why it is not the end of the story

`pmdarima.auto_arima` searches orders by a stepwise algorithm. It is a
reasonable starting point and it is not a substitute for looking.

In [ ]:
try:
    import pmdarima as pm
    auto = pm.auto_arima(train, seasonal=True, m=12, stepwise=True,
                         suppress_warnings=True, error_action="ignore",
                         max_p=3, max_q=3, max_P=2, max_Q=2)
    manual = SARIMAX(train, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
    print(f"  auto_arima chose {auto.order}{auto.seasonal_order}   AIC {auto.aic():8.1f}")
    print(f"  the manual read  (0, 1, 1)(0, 1, 1, 12)   AIC {manual.aic:8.1f}")
except ImportError:
    print("pmdarima is not installed; on Colab run  pip install pmdarima")

**The automatic search lost to the manual reading**, by about four AIC points.

It is not a bug. A stepwise search does not visit every combination, and its
built in tests chose not to difference at all, which sends it down a different
branch of the search tree. Nothing in the output flags that; it simply returns
a model.

**Use automatic selection to generate candidates, then check them against what
the ACF and PACF told you, and compare on AIC yourself.** If the two disagree,
that is information, not an inconvenience.

## 6. Forecast, and score against the bar

In [ ]:
fit = SARIMAX(train, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
pred = np.exp(fit.forecast(12).values)
actual = np.exp(test.values)
snaive = np.exp(train.iloc[-12:].values)

mae = lambda a, p: float(np.mean(np.abs(a - p)))
print(f"  SARIMA               {mae(actual, pred):6.2f} incidents a month")
print(f"  same month last year {mae(actual, snaive):6.2f}")
print(f"  improvement          {100 * (1 - mae(actual, pred) / mae(actual, snaive)):5.0f} percent")

## 7. Seasonal differencing against seasonal terms

Both handle a season, and they assume different things.

| | Assumes | Use it when |
|---|---|---|
| Seasonal **difference**, D = 1 | the seasonal pattern can drift over the years | the pattern is strong and may be changing |
| Seasonal **AR or MA** terms, P or Q | the pattern is stable and can be described by a few coefficients | the pattern looks the same every year |

Using both, as the airline model does, is common and usually safe. Using
neither on a seasonal series is what Module 4 showed failing. **Using two
seasonal differences is over differencing**, and [Module 1](Module_01_Stationarity_Tested.ipynb)
shows how to detect it.

## Exercise

Ashfell is the most seasonal agency in the dataset. Run the same comparison on
Millgate, the least seasonal, and see whether the seasonal terms earn their
place.

In [ ]:
# Fill in the blank, then run.
AGENCY = None              # try "A004"

if AGENCY:
    x = np.log(counts(AGENCY))
    tr, te = x.loc[:"2024-12"], x.loc["2025-01":"2025-12"]
    out = []
    for o, so in [((0, 1, 1), (0, 0, 0, 0)), ((0, 1, 1), (0, 1, 1, 12)),
                  ((0, 1, 1), (1, 0, 0, 12)), ((0, 0, 0), (0, 0, 0, 0))]:
        r = SARIMAX(tr, order=o, seasonal_order=so,
                    trend="c" if (o[1] == 0 and so[1] == 0) else "n",
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        e = float(np.mean(np.abs(np.exp(te.values) - np.exp(r.forecast(12).values))))
        out.append({"order": f"{o} {so[:3]}", "AIC": round(r.aic, 1),
                    "parameters": len(r.params), "forecast error": round(e, 2)})
    print(pd.DataFrame(out).sort_values("AIC").to_string(index=False))
else:
    print("Set AGENCY above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
AGENCY = "A004"
```

**It depends entirely on which column you read, and the two disagree
completely.**

| order | AIC | forecast error |
|---|---|---|
| (0,1,1)(0,1,1) at 12 | **71.7**, best by far | **3.29**, worst |
| (0,1,1)(1,0,0) at 12 | 85.6 | 2.71 |
| (0,1,1) no seasonal | 100.1 | 2.62 |
| (0,0,0), a constant | 102.8, worst | **2.34**, best |

**AIC prefers the seasonal model by nearly thirty points. It forecasts worse
than a constant.** The ranking is almost exactly reversed.

The reason is Millgate's seasonal strength of 0.159, against Ashfell's 0.796.
There is very little genuine seasonal pattern here, so the seasonal difference
and the seasonal moving average term fit the in sample wiggle, which is mostly
noise, and carry none of it usefully into 2025.

Two things follow.

**A seasonal model is not automatically right for monthly data.** Measure the
seasonal strength first, as Intermediate
[Module 5](../../Intermediate/Module_05_Decomposition.md) does, and let that
decide whether the seasonal orders are worth estimating.

**AIC is not a forecast score.** Module 4 made this point with a gap of a
fraction of an incident. Here the gap is large enough to reverse the ranking
outright, which is the strongest form of the same warning: if the deliverable
is a forecast, score forecasts on held out data and do not let a fit statistic
choose for you.

</details>

---

**Next:** [Module 6, Regression with ARMA Errors](Module_06_Regression_With_ARMA_Errors.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*